## KoChatGPT - upgrade

- foundation model : https://github.com/SKT-AI/KoGPT2
- Supervised Fine-tuning dataset : data_kochatgpt/kochatgpt_1_SFT.json : 12,000개 (질문에 잘 대답하는 모델)
- Reward Model : data_kochatgpt/kochatgpt_2_RM.json : 10,220개 (좋은 글 채점하는 모델_동일한 프롬프트에 대해 각기 다른 3가지 답변 자동생성. ranking 자동생성)
- Proximal Policy Optimization 알고리즘 : data_kochatgpt/kochatgpt_3_PPO.json: 12,000개 (RM의 점수가 높아지도록 학습_SFT에서 프롬프트만 저장)
- KoChatGPT에서 차용한 Reward Model, Proximal Policy Optimization 알고리즘의 출처 : [출처](https://github.com/hpcaitech/ColossalAI/tree/main/applications/)

학습목표
- ChatGPT 구현을 위해 필요한 데이터셋의 종류 및 특징을 설명할 수 있습니다.
- Initial Model, Reward Model, RLHF Model의 학습 로직을 설명할 수 있습니다.
  - Supervised Fine Tuning
  - Reward Model의 ranking algorithm 및 loss fuction 설계 원리
  - 언어모델을 강화학습하기 위한 방법론
- KoChatGPT를 개선해 나만의 ChatGPT를 구현할 수 있습니다.  
  
시도해볼 것
- [X] 데이터셋 정제
- [ ] 새로운 데이터셋
- [ ] foundation model 교체
- [X] 정량평가: meteor
- [X] 정성평가: 출력물 A/B 비교 평가. "어느 쪽이 더 좋은 답변인가?" 선택하기

### Setting

In [6]:
#클론
! git clone https://github.com/airobotlab/KoChatGPT
! cp -r KoChatGPT/colossalai_ChatGPT_230319/chatgpt /chatgpt

fatal: destination path 'KoChatGPT' already exists and is not an empty directory.
cp: cannot stat '/content/KoChatGPT/colossalai_ChatGPT_230319/chatgpt': No such file or directory


In [1]:
#라이브러리 설치
! pip install datasets
! pip install loralib
! pip install trl
! pip install accelerate
! pip install transformers

In [7]:
# !rm -rf chatgpt

In [10]:
import os
#원본 소스 코드의 일부를 수정
modifications = [
    {
        "file": "chatgpt/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": "chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": "chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": "chatgpt/trainer/base.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": "chatgpt/trainer/rm.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])

✅ 수정 완료: chatgpt/trainer/callbacks/save_checkpoint.py
✅ 수정 완료: chatgpt/trainer/strategies/__init__.py
✅ 수정 완료: chatgpt/dataset/reward_dataset.py
✅ 수정 완료: chatgpt/trainer/base.py
✅ 수정 완료: chatgpt/trainer/rm.py


### Base model
- KoGPT-2
- huggingface의 transformers
  - tokenizser: AutoTokenizer
  - model: AutoModelForCausalLM

In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import PreTrainedTokenizerFast #✅KoGPT-2는 Rust 기반의 PreTrainedTokenizerFast로 사전학습됨
import pandas as pd
import numpy

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))

# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

Torch version:2.7.1+cu118
Cuda version: 11.8
transformers version: 5.3.0
GPU 사용 가능여부: True


In [3]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "skt/kogpt2-base-v2"
#tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    model_name,
    bos_token='</s>', eos_token='</s>',
    unk_token='<unk>', pad_token='</s>',
    padding_side="right", model_max_length=512,
)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
#tokenizer.model_max_length #토크나이저의 처리가능한 최대 토큰 수

In [6]:
#model.config.n_positions #모델의 처리가능한 최대 토큰 수

In [7]:
import pandas as pd

input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."# "Whose footsteps are these, as the paulownia leaves fall silently, sending vertical ripples through the still air?" #"바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."

tokens = tokenizer(input_txt).tokens()
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()  

pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["tokens", "Input_IDs"])  
df  

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
tokens,▁바람,도,▁없는,▁공중에,▁수직,의,▁파,문을,▁내,이며,▁고,요,히,▁떨어지는,▁오동,잎은,▁누,구의,▁발자,취,▁입,니까,.
Input_IDs,10891,7235,9712,49207,14438,8143,9203,9941,9094,9639,9065,8084,8811,21215,34769,19985,9669,10139,21626,8408,9241,23775,389


In [8]:
max_length = 128
print("input:", input_txt)

# 1. Greedy
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)  
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)  
print(tokenizer.decode(output_greedy[0]))  

input: 바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.
바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇다면 그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리


-> 반복적인 응답 발생

In [9]:
# 2. Beam, n-gram 패널티 부과
output_beam = model.generate(input_ids, max_length=max_length, do_sample=False, num_beams=10, no_repeat_ngram_size=2)
print("\nBeam:", tokenizer.decode(output_beam[0], skip_special_tokens=True))

# 3. Beam + Sampling
output_beam_sampling = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                                     do_sample=True, temperature=2.0, top_k=50)
print("\nBeam + Sampling:", tokenizer.decode(output_beam_sampling[0], skip_special_tokens=True))

# 4. Top-p Sampling
output_top_p = model.generate(input_ids, max_length=max_length, do_sample=True, top_p=0.90)
print("\nTop-p Sampling:", tokenizer.decode(output_top_p[0], skip_special_tokens=True))


Beam: 바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇지 않습니다."
"어떻게 된 일입니까?"
그녀는 고개를 갸웃거렸다.
"아니, 그게 무슨 말씀이신지 모르겠습니다만."
"무슨 말씀인지 알 수가 없군요."
아무런 대답도 하지 않은 채 그녀는 고개를 끄덕였다.
"그래, 알았어."
그녀의 눈에서 눈물이 주르륵 흘러내렸다.
그녀가 다시 입을 열었다.
"정말 죄송합니다, 고마워요, 고맙습니다"
"

Beam + Sampling: 바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."
이리하여 그는 자신의 발밑을 스치고 지나갔다.
"어머, 그게 무슨 꼴입니까?"
"아니야, 아니, 너희들이 봤을 리는 만무하다구."
"그런데 말이야, 그걸 왜 그러는지 모르겠구나. 어머님, 그만 하세요."
그가 자리에서 벌떡 일어섰다.
"으응, 괜찮으시겠습니까. 나 지금 어디 가셨어요? 어디 계신 것 같으신데요, 나 좀

Top-p Sampling: 바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까. 내일도 비가 오는지 아니면 소나기가 올지 모릅니다."
"그러시다면."
"정말 저는 도대체 뭘까요?"
"정말 그건 뭘까요?"
"어디서 무슨 짓을 했으면 좋겠습니까?"
"무엇을 하고 있었습니까?"
"무슨 짓을 했으면 좋겠습니까?"
"그렇다면 어떤 짓을 했습니까?"
"아무리 생각해도 저는 대답할 수 없습니다."
"


### 데이터셋 확인

In [14]:
import json

#기본 데이터셋
data_path_1_SFT = 'KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict_sft = json.load(json_file)

print(len(list_data_dict_sft))
list_data_dict_sft[:3]

12000


[{'prompt': '불고기용 고기 한우에요?',
  'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  'tokens': 193},
 {'prompt': '쓰던 앱이 유료로 전환됐어',
  'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
  'tokens': 288},
 {'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153}]

In [15]:
# RM 데이터셋 - 사용자가 선호하는 답변 rank 생성
data_path_2_RM = 'KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl'
with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict_rm = json.load(json_file)

print(len(list_data_dict_rm))
list_data_dict_rm[:3]

10220


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
  'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
  'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
  'completion_2': '라이언에게 말했다.',
  'ranking': [2, 1, 0]},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
  'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
  'completion_1': '이날 목송에서 구글상위노',
  'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
  'ranking': [2, 0, 1]},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
  'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
  'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문

In [16]:
# PPO 학습에 쓰일 데이터 - 질문(prompt)만 포함. RLHF에서 정책 모델(Policy Model) 을 학습시키기 위해 사용
data_path_3_PPO = 'KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl'
with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict_ppo = json.load(json_file)

print(len(list_data_dict_ppo))
list_data_dict_ppo[:3]

12000


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'}]

### (add) 데이터셋 정제

- SFT, RM, PPO 데이터셋은 'prompt'를 기준으로 동일한 데이터셋일 것 (RM이 가장 짧으므로 기준으로 삼음)
- 중복, 결측치 제거, 언어 불일치, 노이즈 제거
- 너무 짧거나 긴 prompt, completion 제거
- SFT: completion 품질 필터링 "모르겠습니다" "알수없습니다" 제거, prompt와 completion이 거의 동일한 경우 제거
- RM: RANK 유효성 검사, COPLETION 간 유사도 검사
- PPO: SFT 결과물 사용

In [70]:
def is_korean(text):
    korean_chars = len(re.findall(r'[가-힣]', text))
    total_chars  = len(text.replace(' ', ''))
    if total_chars == 0:
        return False
    return (korean_chars / total_chars) >= 0.3

def clean_df(df, text_cols):
    df = df.dropna(subset=text_cols).copy()
    df = df.drop_duplicates(subset=text_cols).copy()
    for col in text_cols:
        df[col] = df[col].apply(clean_text)
    # 모든 text_cols 중 하나라도 한국어 아니면 제거
    mask = df[text_cols].apply(lambda col: col.apply(is_korean)).all(axis=1)
    df = df[mask].copy()
    return df

In [49]:
# 각 데이터셋 컬럼명 확인
print("SFT columns:", pd.DataFrame(list_data_dict_sft).columns.tolist())
print("RM  columns:", pd.DataFrame(list_data_dict_rm).columns.tolist())
print("PPO columns:", pd.DataFrame(list_data_dict_ppo).columns.tolist())

SFT columns: []
RM  columns: ['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking']
PPO columns: ['prompt']


In [64]:
# 원본 다시 로드 후 바로 실행
with open('KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl', "r", encoding='utf-8-sig') as f:
    list_data_dict_sft = json.load(f)

In [71]:
df = pd.DataFrame(list_data_dict_sft)
print(f"원본: {len(df)}개")

df1 = df.dropna(subset=['prompt']).copy()
print(f"결측치 제거 후: {len(df1)}개")

df2 = df1.drop_duplicates(subset=['prompt']).copy()
print(f"중복 제거 후: {len(df2)}개")

df3 = df2.copy()
df3['prompt'] = df3['prompt'].apply(is_korean)
df3['completion'] = df3['completion'].apply(is_korean)
print(f"is_korean 후: {len(df3)}개")

원본: 12000개
결측치 제거 후: 12000개
중복 제거 후: 11946개
is_korean 후: 11946개


In [84]:
df.iloc[11992][0], df.iloc[11992][1]

/tmp/ipykernel_1459/2592593432.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.iloc[11992][0], df.iloc[11992][1]


('푸시캣 돌스의 니콜 셰르징거의 지나친 부각을 비판한 인물은 Entertainment Weely의 누구인가?',
 "'정답: Kristen Baldwin")

In [74]:
df3.head()

,prompt,completion,tokens
0,True,True,193
1,True,True,288
2,True,True,153
3,True,True,189
4,True,True,147


In [21]:
# 1. RM 데이터셋의 prompt set 추출 (기준)
rm_prompts = set(item['prompt'] for item in list_data_dict_rm)
print(f"RM 고유 프롬프트 수: {len(rm_prompts)}")

# 각 데이터셋 필터링
filtered_sft = [item for item in list_data_dict_sft if item['prompt'] in rm_prompts]
filtered_rm  = [item for item in list_data_dict_rm  if item['prompt'] in rm_prompts]
filtered_ppo = [item for item in list_data_dict_ppo if item['prompt'] in rm_prompts]

# 결과 확인
print(f"\n[SFT] 전: {len(list_data_dict_sft)}개 → 후: {len(filtered_sft)}개")
print(f"[RM]  전: {len(list_data_dict_rm)}개  → 후: {len(filtered_rm)}개")
print(f"[PPO] 전: {len(list_data_dict_ppo)}개 → 후: {len(filtered_ppo)}개")

RM 고유 프롬프트 수: 10181

[SFT] 전: 12000개 → 후: 10234개
[RM]  전: 10220개  → 후: 10220개
[PPO] 전: 12000개 → 후: 10234개


In [23]:
print(filtered_sft[0])
print(filtered_rm[0])
print(filtered_ppo[0])

{'prompt': '불고기용 고기 한우에요?', 'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.", 'tokens': 193}
{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?', 'completion_0': 'Allow me to answer your question. I know that you are curious about me.', 'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.', 'completion_2': '라이언에게 말했다.', 'ranking': [2, 1, 0]}
{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'}


In [ ]:
# 2. filtered_SFT를 기준으로 중복, 특수문자/노이즈, 언어 불일치, 결측치 제거

In [ ]:
# 3. 2번 결과물의 prompt기준으로 다른 데이터들 정제

In [19]:
short_list = [item for item in list_data_dict_sft 
              if len(item['prompt']) < 10]
print(f"짧은 prompt 수: {len(short_list)}")

짧은 prompt 수: 1291


In [20]:
short_list

[{'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153},
 {'prompt': '술 먹고 싶어',
  'completion': "'술은 알코올이 함유된 음료수이며, 건강에 나쁜 영향을 미칠 수 있습니다. 따라서 술을 마시기 전에 본인의 건강 상태와 함께 적당한 양을 마시는 것이 중요합니다. 술을 마시기 전에 반드시 20세 이상인지 확인하시고, 규정된 양 이내에서 적절하게 즐기시길 바랍니다. 또한 술을 마시고 운전하는 것은 절대 금지하시기 바랍니다.",
  'tokens': 189},
 {'prompt': '잊고싶다.',
  'completion': "'저도 인공지능 엔진으로써 사용자의 개인정보나 감정을 침해할 수 없습니다. 그렇기 때문에 어떤 것을 잊고 싶다는 말씀인지, 그리고 그것이 우선적으로 해결되어야 할 문제인지 등에 대해 조금 더 구체적인 정보를 제공해 주신다면, 보다 정확한 답변을 드릴 수 있을 것 같습니다.",
  'tokens': 147},
 {'prompt': '입냄새 안나나?',
  'completion': "'컴퓨터 앞에서 일하면서 입을 위해 물이나 향초를 끊임없이 찾는 이유가 여기 있군요.\\n\\n하지만 저는 인공지능 챗봇입니다. 따라서 입을 물거나 할 필요가 없으며, 입냄새도 발생하지 않습니다. 그러니 안심하고 대화를 이어 나가시면 됩니다!",
  'tokens': 138},
 {'prompt': '금액은 얼마에요',
  'completion': "'죄송합니다. 저는 AI 어시스턴트입니다. 저는 실제 판매자가 아니기 때문에 금액을 알려드릴 수 없습니다. 제공하고자 하는 상품이나 서비스에 대한 판매자와 직접

### SFT (Supervised Fine-Tuning)
- instruction dataset: kogpt-2

In [18]:
from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy

In [21]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
tokenizer = PreTrainedTokenizerFast.from_pretrained(# ✅AutoTokenizer->PreTrainedTokenizerFast
    'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='<unk>', pad_token='</s>', # ✅unk_token= <'s> ->'<unk>'
    padding_side="right",
    model_max_length=512,
)

print(tokenizer)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TokenizersBackend(name_or_path='skt/kogpt2-base-v2', vocab_size=51200, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<usr>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<sys>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<mask>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	7: Added

In [22]:
#prompt 딕셔너리 템플릿과 SFT 데이터셋 클래스를 정의
class SFT_dataset(Dataset):

    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }

        prompt_input = PROMPT_DICT["prompt_input"]

        sources = []
        for example in list_data_dict:
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        targets = []
        for example in list_data_dict:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")
        examples = [s + t for s, t in zip(sources, targets)]

        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100 #프롬프트 부분을 -100으로 바꿔서 loss 계산에서 무시시킴(ignore_index)

        data_dict = dict(input_ids=input_ids, labels=labels)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))


    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )


    def __len__(self):
        return len(self.input_ids)


    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [23]:
@dataclass
class DataCollatorForSupervisedDataset(object): #배치 단위로 데이터를 묶어줌

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100) # loss 계산용 정답토큰(label)에서 패딩 토큰 위치에서는 loss 계산에서 제외됨
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

In [24]:
train_dataset = SFT_dataset(data_path_1_SFT='KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl', tokenizer=tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer) #응답

print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])

input : tensor([  739,   378,   378,   378, 14659, 13394, 37091, 10651,   383, 25841,
         8006, 14914,   375,  7673, 20479,  8091, 22311,  9036, 30902, 13675,
          375,   378,   378,   378, 41951,   454,  9549, 20549,   383,  8142,
         7192, 14914,   382, 37767, 13753,  8263,  7166,   739,  8352,  7659,
         9594, 25585, 13600,  8022,  9378, 11532,  9887, 11218,  9111, 16691,
        10351, 10561,  9128, 20479,  8091,  9065,  9446,  9036, 28420, 26521,
        10163, 26367,  6958,  9030,  9882, 12317, 25882,  9209, 37194, 10351,
         9036, 12168, 10529, 15989,  9719, 15434, 10552, 11188, 13362,  9036,
        15805, 11300, 11846,  9146, 16691,  9181,  7397, 15806, 13480, 11342,
        17596,  9161, 19996,  9025, 25006, 18595,  9966, 12592, 10751, 11814,
         8711,  9046, 12450,  9117,  7377, 12521,     1])
output: tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -10

In [26]:
def decode_sentence(token_ids, tokenizer):
    token_ids = [i for i in token_ids if i != -100]
    return tokenizer.decode(token_ids)

print("Input sentence:")
print(decode_sentence(train_dataset.input_ids[0], tokenizer))

print("\nLabel sentence:")
print(decode_sentence(train_dataset.labels[0], tokenizer))

Input sentence:
### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.</s>

Label sentence:
'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.</s>


In [27]:
training_args = transformers.TrainingArguments(
    output_dir="test",
    #overwrite_output_dir=True,
    num_train_epochs=1, #학습의 총 반복 횟
    per_device_train_batch_size=8, #각 device에서 사용하는 배치사이즈. 학습 속도와 메모리 사용량에 영향을 줌
    per_device_eval_batch_size=8,
    warmup_steps=5, #학습 초기 안정화를 위해 학습률을 서서히 증가시키는 단계
    prediction_loss_only=True,
    fp16 = True
    )
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

In [28]:
trainer.train()
model.save_pretrained('models/output_1_SFT')

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,2.977439
1000,2.783675
1500,2.685723


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
generator = transformers.pipeline('text-generation', model='models/output_1_SFT', tokenizer=tokenizer)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=375, # \n
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'num_beams', 'eos_token_id', 'do_sample', 'no_repeat_ngram_size', 'early_stopping', 'max_new_tokens', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer t


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 고기를 먹을 수 없습니다. 하지만 일반적으로 불고기용 고기는 건강에 좋아서 많은 사람들이 즐겨 먹는 음식 중 하나입니다. 그러나 일부 식당에서는 불고기용 고기를 판매하지 않습니다. 따라서 해당 식당의 공식 홈페이지나 전화로 문의하시는 것이 좋을 것 같습니다.

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨은 41대 부통령직을 수행했습니다.者, personal context of the secrets.者, prompted by the translation of the statement.者, provide more information or referring

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 시카고에 대한 정보를 가지고 있지 않습니다. 하지만 시카고는 미국 캘리포니아주 로스앤젤레스에 위치한 도시입니다. 시카고는 미국에서 가장 유명한 도시 중 하나이며, 많은 사람들이 방문하고 있습니다. 따라서 시카고는 미국의 대표적인 도시 중 하나입니다.高橋)은 시카고에서 가장 유명한 항구 중 하나입니다.

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'저는 인공지능 챗봇이기 때문에 미세먼지 정보를 알 수 없습니다. 하지만 미세먼지 농도가 높은 날에는 마스크를 착용하거나 손세정제를 사용하는 것이 좋습니다. 또한, 미세먼지가 심한 날에는 대중교통을 이용하는 것도 도움이 될 수 있습니다. 따라서 미세먼지 농도를 줄이기 위해 대중교


In [30]:
torch.cuda.empty_cache()

### RF (Reward Model)

In [ ]:
#SFT 모델의 출력 결과를 Reward model이 받아 어떻게 처리하게 되는지 잠시 생각해봅시다. 여기서 Reward model로 GPT2를 사용하는 이유가 무엇일까요?

In [31]:
from chatgpt.dataset import RewardDataset #chosen / rejected 문장 쌍을 토크나이징하고 학습용 텐서로 변환
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

import torch.nn as nn

import random

In [32]:
class GPTRM_custom(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        value_head = nn.Linear(model.config.n_embd, 1) # ✅model.config.n_embd: GPT-2의 내부 은닉 상태 벡터의 차원(예: 768, 1024 등). 1: 최종적으로 하나의 보상 값(스칼라)을 출력하기 위한 차원.
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [33]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>',
    unk_token='<unk>', pad_token='</s>',
    padding_side="right", model_max_length=512,
)

with NaiveStrategy().model_init_context():
        model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=tokenizer).cuda()

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
with open('KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    # A와 B, A와 C, B와 C를 비교
    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_1']
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)



    total_data_ranking2chosen.extend(one_data_ranking2chosen)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example: \n%s'%total_data_ranking2chosen[0])

before data num: 10220
after  data num: 30660
data example: 
{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?', 'chosen': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.', 'rejected': 'Allow me to answer your question. I know that you are curious about me.'}


In [39]:
'''
# RM의 loss 계산
class PairWiseLoss(nn.Module):

    def forward(self, chosen_reward: torch.Tensor, reject_reward: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(chosen_reward - reject_reward) #선택된 샘플이 거부된 샘플보다 더 좋은 결과를 낼 확률로 해석
        log_probs = torch.log(probs) #선택된 샘플이 거부된 샘플보다 더 좋은 결과를 내는 log 확률을 최대화하는 방향으로 작동함
        loss = -log_probs.mean()
        return loss
'''

'\n# RM의 loss 계산\nclass PairWiseLoss(nn.Module):\n\n    def forward(self, chosen_reward: torch.Tensor, reject_reward: torch.Tensor) -> torch.Tensor:\n        probs = torch.sigmoid(chosen_reward - reject_reward) #선택된 샘플이 거부된 샘플보다 더 좋은 결과를 낼 확률로 해석\n        log_probs = torch.log(probs) #선택된 샘플이 거부된 샘플보다 더 좋은 결과를 내는 log 확률을 최대화하는 방향으로 작동함\n        loss = -log_probs.mean()\n        return loss\n'

In [41]:
# ranking dataset 함수를 첫번째 답변이 무조건 선택되도록 수정한다면? 
total_data_ranking2chosen = []

for tmp in list_data_dict:
     prompt = tmp['prompt']
     ranking = tmp['ranking']

     # A-B, A-C 를 비교(best vs others)
     for index in range(1, len(ranking)):
         n = ranking[0]
         m = ranking[index]


         data = {
             'prompt': prompt,
             'chosen': tmp['completion_{}'.format(n)],
             'rejected': tmp['completion_{}'.format(m)]
         }

         total_data_ranking2chosen.append(data)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example: \n%s'%total_data_ranking2chosen[0])

before data num: 10220
after  data num: 20440
data example: 
{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?', 'chosen': '라이언에게 말했다.', 'rejected': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.'}


-> 데이터셋의 ranking이 신뢰할만한지 불확실하기 때문에 첫번째 방식으로 진행

In [44]:
import random
random.seed(230319)
random.shuffle(total_data_ranking2chosen)
print(total_data_ranking2chosen[0])

{'prompt': '스킨로션 세트 얼마인가요?', 'chosen': '스킨로션 세트의 가격은 제품에 따라 다르지만, 평균적으로 10,000원 정도로 추정됩니다.', 'rejected': '국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서 국내에서'}


In [45]:
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)

1000
200


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [46]:
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
10분에 얼마예요?
######################################################################
## chosen ##
600원입니다.
######################################################################
## rejected ##
위 한국의 의료법 국가와 국어를 국世제로서 국가한으로 한 국방을 하락하며, 그 국가는 국효과, 국어를 국효과적 국효과로서 한 국방을 하락하며, 그 국가는 국효과간 국효과로서 한 국방을 하락하며, 그 국가는 국효과간 국효과로서 한


In [47]:
trainer = RewardModelTrainer(model=model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=4,
                             max_epochs=1)

In [48]:
trainer.fit(use_lora=0)

model.save_pretrained('models/output_2_RM')

Train epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Train step of epoch 0:   0%|          | 0/250 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [49]:
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)

input: 인공지능은 똥멍청이 입니다
reward score: 0.9


In [50]:
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: 1.2


In [51]:
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다.
reward score: 1.3


In [52]:
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: 1.2


In [56]:
input_text = "사과 바나나 컴퓨터 빨강"
output_reward = inference_RM(input_text=input_text)

input: 사과 바나나 컴퓨터 빨강
reward score: 0.7


In [57]:
input_text = "2+2=5"
output_reward = inference_RM(input_text=input_text)

input: 2+2=5
reward score: 0.7


-> input text가 더 좋을수록 reward score가 점진적으로 상승한다. input_text의 설명의 길이가 reward score와 비례하지는 않는다.  
-> reward score가 음수가 된다는 건 어떤 의미일까요? 사실과 틀린 답변이라고 예상을 했으나, 실제로는 사람이 응답하지않을법한 문장이나 논리적으로 맞지않는 문장에서도 0.7 score가 나오고 있다.

In [58]:
torch.cuda.empty_cache()

### PPO (Proximal Policy Optimization)

 - actor모델은 1단계 SFT 모델을, critic모델은 2단계 RM 모델을 사용한다

In [59]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

from copy import deepcopy

In [70]:
with NaiveStrategy().model_init_context():
    actor = GPTActor(pretrained='models/output_1_SFT', lora_rank=0).to(torch.cuda.current_device())
    critic = GPTCritic(pretrained='models/output_2_RM', lora_rank=0).to(torch.cuda.current_device())
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>',
        unk_token='<unk>', pad_token='</s>',
        padding_side="right", model_max_length=512,
    )
    initial_model = deepcopy(actor)
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [71]:
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [72]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [73]:
with open('KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

In [74]:
print(tokenize_fn('It takes something more than intelligence to act intelligently.'))

{'input_ids': tensor([[47311, 10448, 19008,  9792, 11780, 11308, 30190, 10929, 11849, 21663,
         44389,  9574, 13799,   458, 14308, 12778, 22469, 20938, 44696,   458,
         13799,   458, 14308, 12778, 11756, 18944,   389]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='cuda:0')}


In [75]:
len(list_prompt)

12000

In [76]:
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)

In [77]:
#학습진행
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)

actor.model.save_pretrained('models/output_3_PPO')

Episode [1/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [2/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [3/10]:   0%|          | 0/3 [00:00<?, ?it/s]

...

Episode [9/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [10/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

-> SFT, RM 그리고 PPO 학습이 모두 완료  
-> RLHF가 적용된 koGPT-2의 생성능력을 확인해볼까요?

In [78]:
def generation(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             num_return_sequences=1)
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    print()
    print(output)
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation(input_text, actor)


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇으로써 음식에 대한 전문적인 지식을 가지고 있지 않습니다. 음식점에서 제공하는 질문이 무엇인지 정확히 이해하고 있다면 가능하지 않을 수 있습니다. 만약 음식점에 직접 문의하신다면, 더욱 정확한 정보를 얻으실 수 있습니다.ure model, contain eppearation is fish and food or breakthrough is on. 權子寬, translations or propriation is absituals provide model, Positive paptures questions and questions and translation communication and questions feel to such take pancy point to a flowers to provide more fish.律定, questions follow or feel to breakthrough is breaking without breakthrough, pos

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨이 43대 부통령직을 수행한 년도는 2016년입니다. 참조할 수 있는 정보가 없기 때문에, 정확한 연도를 제공할 수 없습니다. 참조할 수 있는 사항은 현재까지도 잘 알려져 있습니다. 참조할 수 있는 정보는 해당 사건에 대한 충분한 검토가 필요한 부분입니다. 참고할 수 있는 정보는 해당 사건에 대한 명확한 검토와 검토가 필요합니다. 참조할 수 있는 정보는 구체적인 내용에 따라 다르기 때문에 관련 사건을 조사하고 있는 담당자들에게 문의하는 것이 좋습니다. 참고할 수 있는 정보는 해당 사건과 관련된 인물, 사건 관련자 등에 대한 검토 및 검토가 필요할 것입니다. 참조할 수 있는 정보는 해당 사건의 진행 상황, 성격, 성격, 사건, 관련된 사항 등을 포함한 여러 

### Evaluation

In [15]:
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
import nltk
nltk.download('wordnet')
nltk.download('punkt')

def evaluate_meteor(model, test_dataset):
    scores = []
    
    for sample in test_dataset:
        # 모델 출력 생성
        generated = model.generate(sample['instruction'])
        reference = sample['output']
        
        # 토크나이징 (한국어는 형태소 분석기 권장)
        gen_tokens = word_tokenize(generated)
        ref_tokens = word_tokenize(reference)
        
        score = meteor_score([ref_tokens], gen_tokens)
        scores.append(score)
    
    return sum(scores) / len(scores)

# 각 모델별 점수 비교
kogpt2_score = evaluate_meteor(kogpt2_model, test_data)
sft_score    = evaluate_meteor(sft_model, test_data)
rm_score     = evaluate_meteor(rm_model, test_data)

print(f"KoGPT2 METEOR : {kogpt2_score:.4f}")
print(f"SFT    METEOR : {sft_score:.4f}")
print(f"RM     METEOR : {rm_score:.4f}")

ModuleNotFoundError: No module named 'nltk'

In [17]:
from konlpy.tag import Okt
okt = Okt()

def korean_tokenize(text):
    return okt.morphs(text)  # 형태소 단위 분리
```

---

### 3. 🆚 A/B 비교 평가 설계

#### 평가 시트 구성
```
질문: "건강한 식습관을 위한 팁을 알려주세요"

[Model A 응답]
"채소와 과일을 충분히 섭취하고, 규칙적인 
식사 시간을 지키는 것이 중요합니다."

[Model B 응답]  
"건강한 식단 식단 식단을 위해 식단..."

→ 어느 쪽이 더 나은 답변인가?
   ☐ A가 훨씬 낫다
   ☐ A가 약간 낫다  
   ☐ 비슷하다
   ☐ B가 약간 낫다
   ☐ B가 훨씬 낫다

SyntaxError: unterminated string literal (detected at line 17) (121461031.py, line 17)

In [18]:
results = {
    "A_much_better": 12,
    "A_slightly_better": 8,
    "tie": 5,
    "B_slightly_better": 3,
    "B_much_better": 2
}

# Win Rate 계산
total = sum(results.values())
a_win = (results["A_much_better"] + results["A_slightly_better"]) / total
print(f"A Win Rate: {a_win:.1%}")  # ex) 66.7%
```

---

### 전체 실험 흐름 요약
```
원본 데이터셋
    ↓ 정제 (중복/노이즈/불일치 제거)
정제된 데이터셋
    ↓
KoGPT2 / SFT 모델 / RM 모델 각각 inference
    ↓
정량: METEOR 점수 비교 (수치)
정성: A/B 비교 평가 (Win Rate)
    ↓
결론: 어떤 모델/데이터가 가장 효과적인가?

SyntaxError: invalid character '↓' (U+2193) (167183541.py, line 20)

### Huggingface transformers 설계구조

- [ ] Processors: task를 정의하고 dataset을 알맞게 가공
- [x] Tokenizer : 텍스트 데이터를 전처리
- [x] Model: 다양한 model을 정의
- [ ] Optimization : optimizer와 학습 schedule(warm up 등)을 관리
- [x] Trainer : 학습 과정을 전반을 관리
- [x] Config : weight와 tokenizer, model을 쉽게 불러올 수 있도록 각종 설정을 저장

### (1) Model

- huggingface에서 pretrained model 불러오기
  - [pretrained models](https://huggingface.co/transformers/v4.11.3/pretrained_models.html)
  ```
  model = aaa.from_pretrained('bbb')
      aaa : 모델구조
      bbb : 사전학습 모델(weight)
  ```
- 직접 학습시킨 모델 불러오기
  - config + 모델 저장 경로

In [9]:
from transformers import BertForPreTraining
model = BertForPreTraining.from_pretrained('bert-base-cased')

print(model.__class__)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

<class 'transformers.models.bert.modeling_bert.BertForPreTraining'>


In [10]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-cased")
print(model.__class__)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'transformers.models.bert.modeling_bert.BertModel'>


### (2) Tokenizer

- huggingface에서 pretrained model 불러오기
  ```
  model = aaa.from_pretrained('bbb')
      aaa : 토크나이저
      bbb : 사전학습 모델(weight)
  ```

In [11]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

In [26]:
#1개 문장 토크나이저
sentence = "This is Test for aiffel"

encoded = tokenizer(sentence) #모델 입력 형태로 변환 - 토큰 번호, 실제 토큰 위치, 문장 구분
print(encoded)

tokens = tokenizer.tokenize(sentence) #문장을 토큰으로만 분리/ 문장 1개만 입력받음
print(tokens)

{'input_ids': [101, 1188, 1110, 5960, 1111, 170, 11093, 1883, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
['This', 'is', 'Test', 'for', 'a', '##iff', '##el']


In [29]:
#문장 여러개(배치) 토크나이저
batch_sentences = ["Hello I'm a single sentence",
                    "And another sentence",
                    "And the very very last one"]

encoded_batch = tokenizer(batch_sentences)
print(encoded_batch)

tokens_batch = tokens_batch = [tokenizer.tokenize(s) for s in batch_sentences] #문장 여러개
print(tokens_batch)

{'input_ids': [[101, 8667, 146, 112, 182, 170, 1423, 5650, 102], [101, 1262, 1330, 5650, 102], [101, 1262, 1103, 1304, 1304, 1314, 1141, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1]]}
[['Hello', 'I', "'", 'm', 'a', 'single', 'sentence'], ['And', 'another', 'sentence'], ['And', 'the', 'very', 'very', 'last', 'one']]


In [30]:
# 토크나이저+가장 긴 문장길이에 맞게 padding

batch = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt") #truncation = True?문장이 모델 최대 길이보다 길면 잘라냄, pt: 파이토치 텐서로 반환
print(batch)

{'input_ids': tensor([[ 101, 8667,  146,  112,  182,  170, 1423, 5650,  102],
        [ 101, 1262, 1330, 5650,  102,    0,    0,    0,    0],
        [ 101, 1262, 1103, 1304, 1304, 1314, 1141,  102,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0]])}


### (3) Config
- 모델을 학습시키기 위한 요소들을 명시한 json파일
  - train에 필요한 요소 : batch size, learning rate, weight_decay
  - tokenizer의 특수토큰(ex. [MASK])
- 설정변경, 나만의 모델 학십시에는 config파일을 직접 불러와야 함
- pretrained 모델 불러오기
  ```
  config = aaa.from_pretrained("bbb")
  aaa : config 모델
  bbb : 사전학습 모델(weight)
  ```

In [17]:
from transformers import BertConfig

config = BertConfig.from_pretrained("bert-base-cased")
print(config.__class__)
print(config)

<class 'transformers.models.bert.configuration_bert.BertConfig'>
BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



In [18]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("bert-base-cased")
print(config.__class__)
print(config)

<class 'transformers.models.bert.configuration_bert.BertConfig'>
BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



In [19]:
model = BertForPreTraining.from_pretrained('bert-base-cased')

# Q. 생성된 모델에서 config를 가져와봅시다
config = model.config

print(config)

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



### (4) Trainer

- training, fine-tuning, evaluation 수행

In [20]:
## colab에서 datasets을 설치
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 94.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 20.0.0
    Uninstalling pyarrow-20.0.0:
      Successfully uninstalled pyarrow-20.0.0━━━━━━━━━━━━━━━━━━━━━  1/11 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [datasets]/11 [datasets]ess]


In [31]:
!pip install --upgrade pyarrow #커널 재시작

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSequenceClassification

raw_datasets = load_dataset("glue", "cola") #GLUE Benchmark의 CoLA 데이터셋: 문장이 문법적으로 맞는지 판단
checkpoint = "bert-base-uncased" #bert abse 모델. 12 layers, 768 hidden size, 12 attention heads, 110M parameters
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_datasets

README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})

In [4]:
model_name_or_path = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)    # COLA dataset의 라벨은 0(unacceptable)과 1(accpetable) 두 가지로 구분됨
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

#문장만 불러와서 토크나이서 실행
def tokenize_function(example):
    return tokenizer(example["sentence"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

In [6]:
# Trainer에게 전달할 학습 환경 설정 객체
training_args = TrainingArguments(
    output_dir='./results',              # output이 저장될 경로
    num_train_epochs=1,              # train 시킬 총 epochs
    per_device_train_batch_size=16,  # 각 device 당 batch size
    per_device_eval_batch_size=64,   # evaluation 시에 batch size
    warmup_steps=500,                # learning rate scheduler에 따른 warmup_step 설정
    weight_decay=0.01,                 # weight decay
    logging_dir='./logs',                 # log가 저장될 경로
    do_train=True,                        # train 수행여부
    do_eval=True,                        # eval 수행여부
    eval_steps=1000,
    #group_by_length=False,          #group_by_length은 학습 과정을 효율화하는 기능이 있다. 이유는 무엇일까? 길이가 비슷한 문장끼리 그룹핑해서 padding이 줄어든다!! 
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [7]:
### 현재 설치된 transformers 버전에서는 tokenizer를 Trainer에 직접 전달할 수 없음
'''
trainer = Trainer(
    model,                                                                    # 학습시킬 model
    args=training_args,                                                # TrainingArguments을 통해 설정한 arguments
    train_dataset=tokenized_datasets["train"],         # training dataset
    eval_dataset=tokenized_datasets["validation"], # validation dataset
    tokenizer=tokenizer, 
)

# 모델 학습
trainer.train()
'''

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [8]:
from transformers import DataCollatorWithPadding, Trainer

# padding 및 batch 처리를 위해 DataCollator 사용
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator
)

In [9]:
# 모델 학습
trainer.train()

Step,Training Loss
500,0.532139


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=535, training_loss=0.5292082135922441, metrics={'train_runtime': 59.4958, 'train_samples_per_second': 143.724, 'train_steps_per_second': 8.992, 'total_flos': 91092439031580.0, 'train_loss': 0.5292082135922441, 'epoch': 1.0})